# 11 — Dynamic tool spawning

**Goal:** let an agent (or another part of the system) **create a brand-new tool at runtime** by emitting code, then run it safely in a sandbox.

orqest has long been able to spawn **agents** at runtime (`AgentFactory.spawn(AgentSpec)`). But each spawned agent could only use tools that were **already registered** in `ToolRegistry`. If an LLM asked for a tool that did not exist, the request was silently dropped.

This notebook closes the loop. The pieces:

1. **`Sandbox`** — runs untrusted Python safely. Two backends: in-process (Tier 0) and subprocess (Tier 1, the production default).
2. **`GeneratedToolSpec`** — a spec that includes an `implementation: str` field. The body is real Python code.
3. **`DynamicToolFactory`** — turns a `GeneratedToolSpec` into a runnable `pydantic_ai.Tool` after validating the code.
4. **`BaseAgent.add_tool()`** — attaches a newly-spawned tool to an existing agent at runtime.

By the end you will have an agent that uses a tool which **did not exist when the agent was created**.

**Cost: ~$0.50** (one short LLM call at the end).


## 1. The sandbox

The sandbox is the **safety boundary**. It validates code (static AST check) and then runs it. Two backends ship out of the box:

- **`InProcessSandbox`** (Tier 0) — runs code in the current Python process. Fast but **unsafe**: a `while True` will hang the host. Requires `unsafe=True` to construct, so it is impossible to use accidentally.
- **`SubprocessSandbox`** (Tier 1) — runs code in a fresh subprocess with memory and CPU limits, and an outer timeout. This is the production default.

Both backends follow the same `Sandbox` protocol: `validate(code)` then `execute(code, args)`.


In [1]:
import asyncio
from orqest.sandbox import InProcessSandbox, SubprocessSandbox, ValidationError


# Tier 0 — fast, no isolation. Requires unsafe=True.
inproc = InProcessSandbox(unsafe=True)
result = await inproc.execute(
    "return args['x'] + args['y']",
    args={'x': 6, 'y': 7},
    allowed_imports=set(),
)
print(f"in-process: success={result.success} output={result.output} dur={result.duration_ms:.2f}ms")


# Tier 1 — fresh subprocess + RLIMIT_AS + RLIMIT_CPU + outer timeout.
sub = SubprocessSandbox()
result = await sub.execute(
    "import re; return re.findall(r'\\d+', args['text'])",
    args={'text': 'a1 b22 c333'},
    allowed_imports={'re'},
    timeout_s=2.0,
)
print(f"subprocess: success={result.success} output={result.output} dur={result.duration_ms:.0f}ms")

in-process: success=True output=13 dur=0.10ms
subprocess: success=True output=['1', '22', '333'] dur=67ms


### 1b. Validation — default-deny imports

The sandbox runs a static AST walk **before** executing anything. It refuses any code that:

- imports a module not listed in `allowed_imports` (which defaults to **empty**),
- calls forbidden names like `eval`, `exec`, `__import__`, `open`,
- accesses dunder attributes (`__class__`, `__subclasses__`, etc.) — these are the classic Python sandbox-escape tricks.

The behaviour is identical across both backends, so a piece of code that validates in Tier 0 also validates in Tier 1.


In [2]:
try:
    await sub.validate('import os; return os.getcwd()', allowed_imports=set())
except ValidationError as exc:
    print(f'rejected: {exc}')

try:
    await sub.validate('return eval("1+1")', allowed_imports=set())
except ValidationError as exc:
    print(f'rejected: {exc}')

rejected: line 1: import 'os' not in allowed_imports (empty) (Import)
rejected: line 1: call to forbidden name 'eval' (Call); line 1: reference to forbidden name 'eval' (Name)


### 1c. Execute-time timeout

Validation catches **static** bad behaviour. But what about an infinite loop? The subprocess sandbox handles that with an outer `asyncio.wait_for` and a `RLIMIT_CPU` cap.


In [3]:
result = await sub.execute(
    'while True: pass',
    args={},
    allowed_imports=set(),
    timeout_s=1.0,
)
print(f"timeout: success={result.success} error={(result.error or '')[:80]}")

timeout: success=False error=sandbox execution timed out after 1.00s


## 2. `DynamicToolFactory` — spec to live tool

A `GeneratedToolSpec` describes a tool whose implementation is Python source. The factory validates the source, then wraps it in a `pydantic_ai.Tool` whose body invokes `sandbox.execute()` at call time.

The result is a regular `pydantic_ai.Tool`. It can be bound to any agent. Nothing downstream knows or cares that the tool was generated at runtime.


In [4]:
from orqest.autonomy import DynamicToolFactory, GeneratedToolSpec

factory = DynamicToolFactory(SubprocessSandbox())

doi_spec = GeneratedToolSpec(
    name='extract_dois',
    description='Extract DOIs from a text blob.',
    parameters={'text': {'type': 'string'}},
    implementation=(
        "import re\n"
        "matches = re.findall(r'10\\.\\d{4,}/[\\w.\\-/]+', args['text'])\n"
        "return {'dois': matches, 'count': len(matches)}\n"
    ),
    allowed_imports={'re'},
)

doi_tool = await factory.spawn(doi_spec)
print(f'spawned tool: {doi_tool.name} - {doi_tool.description}')


# Call it directly to see it work.
sample = 'See Smith (2024) at 10.1234/foo, also Jones (2025) at 10.5678/bar/baz.'
out = await doi_tool.function(text=sample)
print(f'invoked: {out}')

spawned tool: extract_dois - Extract DOIs from a text blob.
invoked: {'dois': ['10.1234/foo', '10.5678/bar/baz.'], 'count': 2}


## 3. `AgentFactory` dispatch — mixed registered + generated tools

An `AgentSpec` is the declarative description of an agent. Its `tools` field can mix two kinds:

- **`ToolSpec`** — a reference to a pre-registered tool by name.
- **`GeneratedToolSpec`** — a fresh tool to be generated by the factory.

`AgentFactory.spawn` resolves both kinds. Registered tools come from the registry. Generated tools are run through the `DynamicToolFactory`. The agent ends up with a flat tool list, no matter where each tool came from.


In [5]:
from pydantic_ai import Tool
from pydantic_ai.models.test import TestModel
from orqest.autonomy import AgentFactory, AgentSpec, ToolSpec, ToolRegistry


# Pre-register a tool the conventional way.
async def search(q: str) -> str:
    """Search the web."""
    return f'searched: {q}'


registry = ToolRegistry()
registry.register(Tool(search, name='search'))

agent_factory = AgentFactory(registry=registry, tool_factory=factory)


spec = AgentSpec(
    name='researcher',
    system_prompt='You research topics and extract DOIs.',
    output_schema={
        'type': 'object',
        'properties': {'answer': {'type': 'string'}},
        'required': ['answer'],
    },
    tools=[
        ToolSpec(name='search', description='Search the web'),           # registered
        GeneratedToolSpec(                                                # generated
            name='extract_dois',
            description='Extract DOIs from text.',
            parameters={'text': {'type': 'string'}},
            implementation=doi_spec.implementation,
            allowed_imports={'re'},
        ),
    ],
)


agent = agent_factory.spawn(spec, model=TestModel())
print(f'agent tools: {sorted(t.name for t in agent.tools)}')

agent tools: ['extract_dois', 'search']


## 4. `add_tool` — attach a tool to an existing agent

What if the agent already exists and you want to give it a new tool?

`BaseAgent.add_tool()` does this. It appends the tool to `self.tools` and **invalidates the cached internal `pydantic_ai.Agent`**. The next time the agent is used, the underlying agent is rebuilt with the new tool list.

This is important: without the cache invalidation, the new tool would never be visible to the LLM.


In [6]:
# Spawn a fresh agent with NO tools.
bare_agent = agent_factory.spawn(
    AgentSpec(
        name='bare',
        system_prompt='you do nothing initially',
        output_schema={
            'type': 'object',
            'properties': {'answer': {'type': 'string'}},
            'required': ['answer'],
        },
        tools=[],
    ),
    model=TestModel(),
)
print(f'before: tools={[t.name for t in bare_agent.tools]} _agent cached? {bare_agent._agent is not None}')

# Force the internal agent to be built so we can see the cache change.
_ = bare_agent.agent
print(f'after access:   _agent cached? {bare_agent._agent is not None}')

# Hand the agent a freshly-spawned tool.
bare_agent.add_tool(doi_tool)
print(f'after add_tool: tools={[t.name for t in bare_agent.tools]} _agent cached? {bare_agent._agent is not None}')

# Next access rebuilds with the new tool list.
_ = bare_agent.agent
print(f'after rebuild:  _agent rebuilt with {[t.name for t in bare_agent.tools]}')

before: tools=[] _agent cached? False
after access: _agent cached? True
after add_tool: tools=['extract_dois'] _agent cached? False
after rebuild: _agent rebuilt with ['extract_dois']


## 5. End-to-end — a real LLM agent uses a runtime-spawned tool

Now we put it all together with a real LLM (not the test model). The flow:

1. Build an agent without any tools.
2. Spawn a `GeneratedToolSpec` for DOI extraction at runtime.
3. Attach the tool to the agent.
4. Ask the agent to extract DOIs from a text. The agent should call the tool.

The agent has **no idea** the tool was generated. As far as it knows, it has a normal pydantic-ai tool called `extract_dois`.


In [7]:
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()


class DoiAnswer(BaseModel):
    answer: str = Field(description='Comma-separated list of DOIs found, or "none".')


class DoiAgent(BaseAgent[GlobalState, DoiAnswer]):
    async def _run_implementation(self, state, **kwargs):
        latest = state.get_latest_message('user') or ''
        result = await self.call_model(latest, state)
        return result.output


# 1. Spawn an agent with no tools.
real_agent = DoiAgent(
    agent_name='doi_finder',
    system_prompt=(
        'You find DOIs in user-supplied text. Use the extract_dois tool '
        'to scan the text, then report the DOIs found in your answer.'
    ),
    output_type=DoiAnswer,
    model=config.llm_model,
    api_key=config.llm_api_key,
)

# 2 + 3. Attach the runtime-spawned DOI tool.
real_agent.add_tool(doi_tool)
print(f'real_agent tools: {[t.name for t in real_agent.tools]}')

# 4. Run.
state = GlobalState()
state.add_message('user', f'Find the DOIs in this text:\n\n{sample}')
result = await real_agent.run(state)
print(f'final answer: {result.answer}')

real_agent tools: ['extract_dois']


final answer: 10.1234/foo, 10.5678/bar/baz.


## 6. Observability — bus events

The `DynamicToolFactory` publishes lifecycle events to an `EventBus` when one is provided. These are the events to subscribe to in production:

- **`tool.spawned`** — a `GeneratedToolSpec` was validated and turned into a `Tool`.
- **`tool.invocation_completed`** / **`tool.invocation_failed`** — the tool ran.
- **`sandbox.validation_rejected`** — the spec was rejected at validation time.

Below we spawn one good tool and one bad tool, then look at the events that fired.


In [8]:
from orqest.observability.events import EventBus

bus = EventBus()
events = []
for et in ('tool.spawned', 'tool.invocation_completed', 'tool.invocation_failed', 'sandbox.validation_rejected'):
    bus.subscribe(et, events.append)


bus_factory = DynamicToolFactory(SubprocessSandbox(), bus=bus)


# Good spawn + invoke.
spec_ok = GeneratedToolSpec(
    name='reverse',
    description='Reverse a string.',
    parameters={'text': {'type': 'string'}},
    implementation='return args["text"][::-1]',
    allowed_imports=set(),
)
tool_ok = await bus_factory.spawn(spec_ok)
await tool_ok.function(text='hello')


# Bad spawn — disallowed import.
try:
    await bus_factory.spawn(GeneratedToolSpec(
        name='bad',
        description='x',
        parameters={},
        implementation='import os; return os.getcwd()',
        allowed_imports=set(),
    ))
except ValidationError:
    pass


await asyncio.sleep(0.05)        # let bus tasks flush
for e in events:
    print(f'  {e.event_type}: {e.data}')

  tool.spawned: {'tool_name': 'reverse'}
  tool.invocation_completed: {'tool_name': 'reverse', 'duration_ms': 168.75600399998802}
  sandbox.validation_rejected: {'tool_name': 'bad', 'reason': "line 1: import 'os' not in allowed_imports (empty) (Import)"}


## Recap

The whole pattern:

1. The **sandbox** is the safety boundary. Static AST validation + a runtime subprocess with limits.
2. A **`GeneratedToolSpec`** describes a tool whose body is Python source.
3. The **`DynamicToolFactory`** validates and wraps it as a `pydantic_ai.Tool`.
4. The **`AgentFactory`** can mix registered and generated tools in a single spec.
5. **`BaseAgent.add_tool`** attaches tools at runtime and invalidates the cached internal agent.
6. Lifecycle events on the bus give you full visibility.

**Why this matters.** It closes the last gap on the autonomy ladder: agents that not only **plan** their work but also **create the tools** they need to do it. With the sandbox as the safety boundary, this is safe in production.

**What's next:**

- **Procedural-memory persistence.** A spawned tool that proves itself becomes a `Skill` in `LocalMemoryStore`. Future runs recall it without re-asking the LLM.
- **Pooled subprocess sandbox.** Reuse pre-warmed subprocesses to amortize the ~50ms startup cost.
- **Stronger isolation backends.** Docker or E2B for higher-trust scenarios.

See `docs/concepts/sandbox.md` and the "Dynamic tool spawning" section of `docs/concepts/autonomy.md`.
